## 1. Chuẩn bị môi trường

Bật GPU và Internet trên Kaggle. Chạy phần clone một lần khi bắt đầu phiên làm việc.


In [ ]:
!git clone https://github.com/tranminhduc9/well-log-imputation.git /kaggle/working/well-log-imputation
%cd /kaggle/working/well-log-imputation


In [ ]:
%env CUBLAS_WORKSPACE_CONFIG=:4096:8
%pip install -q -r requirements.txt


## 2. Đường dẫn và cấu hình

Kết quả lưu trực tiếp trong `/kaggle/working/models/`: checkpoint theo `<model>/seed_<seed>/`, các bảng CSV/JSON và biểu đồ dùng tên cố định. Chạy lại sẽ ghi đè file cùng tên; file tải xuống là `/kaggle/working/models.zip`. Không tạo thư mục theo ngày giờ.


In [ ]:
from argparse import Namespace
from pathlib import Path
import gc
import json
import logging
import shutil
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import FileLink, display

In [ ]:
PROJECT_ROOT = Path('/kaggle/working/well-log-imputation')
DATA_ROOT = Path('/kaggle/input/datasets/minhduc0912/geolink-dataset')
OUTPUT_ROOT = Path('/kaggle/working/models')
DEVICE = 'cuda'
MODELS = ('locf', 'xgboost', 'brits', 'saits', 'conv_saits')
NEURAL_MODELS = ('brits', 'saits', 'conv_saits')

In [ ]:
sys.path.insert(0, str(PROJECT_ROOT))
from src.data.loader import DataLoader
from src.experiments.runner import run_experiment
from src.experiments.inference import restore_run
from src.models._training import synchronized_time

In [ ]:
SEEDS = (912, 219)
NEURAL_EPOCHS = 500
NEURAL_PATIENCE = 30
MODEL_SETTINGS = {
    "xgboost": {"max_depth": 3, "learning_rate": 0.03, "n_estimators": 300},
    "brits": {
        "hidden_size": 64, "learning_rate": 0.0005, "batch_size": 32,
        "consistency_weight": 0.1, "weight_decay": 1e-5,
        "mit_weight": 1.0, "mit_reduction": "segment",
        "masking_strategy": "mixed", "gradient_clip": 1.0,
        "validation_metric": "rmse", "min_delta": 0.0,
    },
    "saits": {
        "d_model": 256, "d_inner": 128, "n_layers": 2, "n_heads": 4,
        "learning_rate": 0.0005, "batch_size": 32,
        "ort_weight": 1.0, "mit_weight": 1.0, "mit_reduction": "segment",
        "min_delta": 0.0,
    },
    "conv_saits": {
        "d_model": 256, "d_inner": 128, "n_layers": 2, "n_heads": 4,
        "learning_rate": 0.0005, "batch_size": 32,
        "encoder_channels": 8, "kernel_size": 15, "conv_expansion": 2,
        "multi_scale_conv": True, "multi_scale_kernels": (3, 7, 15),
        "depth_encoding": False,
        "masking_strategy": "mixed", "mit_reduction": "segment",
        "ort_weight": 1.0, "mit_weight": 1.0, "min_delta": 0.0,
        "masking_rate": 0.2, "independent_shuffle": True,
    },
}

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(message)s', force=True)


## 3. Huấn luyện và đánh giá

In [ ]:
args = Namespace(
    data_root=str(DATA_ROOT), output_dir=str(OUTPUT_ROOT), device=DEVICE,
    models=MODELS, seeds=SEEDS, evaluate_test=True,
    xgb_estimators=MODEL_SETTINGS['xgboost']['n_estimators'],
    brits_epochs=NEURAL_EPOCHS, brits_patience=NEURAL_PATIENCE,
    saits_epochs=NEURAL_EPOCHS, saits_patience=NEURAL_PATIENCE,
    conv_saits_epochs=NEURAL_EPOCHS, conv_saits_patience=NEURAL_PATIENCE,
)
output_dir = run_experiment(args, PROJECT_ROOT, MODEL_SETTINGS)

summary = pd.read_csv(output_dir / 'summary.csv')
history = pd.read_csv(output_dir / 'training_history.csv')
manifest = json.loads((output_dir / 'experiment_results.json').read_text())
preprocessing = json.loads((output_dir / 'preprocessing_parameters.json').read_text())
runs = pd.DataFrame(manifest['runs'])
neural_history = history[history['model'].isin(NEURAL_MODELS)]

print('Results:', output_dir)
display(summary[['label', 'split', 'scenario', 'n_runs',
                 'mae_mean', 'mae_std', 'rmse_mean', 'rmse_std']])


## 4. Hội tụ theo epoch

Biểu đồ cho BRITS, SAITS và Conv-SAITS. Dấu chấm đánh dấu checkpoint được chọn trên validation.


In [ ]:
neural_runs = runs[runs['model'].isin(NEURAL_MODELS)]
for seed in SEEDS:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for run in neural_runs[neural_runs['seed'] == seed].itertuples():
        curve = neural_history[
            (neural_history['model'] == run.model) & (neural_history['seed'] == seed)
        ].sort_values('epoch')
        axes[0].plot(curve['epoch'], curve['loss'], label=run.label)
        line, = axes[1].plot(curve['epoch'], curve['validation_rmse'], label=run.label)
        best = curve[curve['epoch'] == run.best_epoch]
        axes[1].scatter(best['epoch'], best['validation_rmse'], color=line.get_color())

    axes[0].set(title=f'Train loss / seed {seed}', xlabel='Epoch', ylabel='Loss')
    axes[1].set(title=f'Validation RMSE / seed {seed}', xlabel='Epoch', ylabel='Normalized RMSE')
    for ax in axes:
        ax.legend(fontsize=8)
        ax.grid(alpha=0.2)
    fig.tight_layout()
    fig.savefig(output_dir / f'convergence_seed_{seed}.png', dpi=150)
    plt.show()
    plt.close(fig)


## 5. Parameters và thời gian chạy

In [ ]:
WARMUP, REPEATS, N_SAMPLES = 3, 5, 8
samples_by_scenario = {
    scenario: DataLoader(DATA_ROOT, split='test', scenario=scenario).load()['X'][:N_SAMPLES]
    for scenario in preprocessing['missing_scenarios']
}
parameter_counts = {'locf': 0, 'xgboost': np.nan}
inference_rows = []

for run in runs.to_dict('records'):
    name, seed = run['model'], run['seed']
    device = 'cpu' if name == 'locf' else DEVICE
    model = restore_run(
        {**run, 'artifact': str(output_dir / run['artifact'])}, device=device, batch_size=32,
    )
    if name in NEURAL_MODELS:
        parameter_counts[name] = sum(p.numel() for p in model.backend.network.parameters())

    for scenario, samples in samples_by_scenario.items():
        elapsed = []
        for sample in samples:
            inputs = {'X': sample[None]}
            for _ in range(WARMUP):
                model.impute(inputs)
            for _ in range(REPEATS):
                start = synchronized_time(device)
                model.impute(inputs)
                elapsed.append(1000 * (synchronized_time(device) - start))
        inference_rows.append({
            'model': name, 'seed': seed, 'scenario': scenario,
            'samples': len(samples), 'inference_ms_per_sample': np.mean(elapsed),
        })
    del model
    gc.collect()
    torch.cuda.empty_cache()

inference = pd.DataFrame(inference_rows)
inference.to_csv(output_dir / 'inference_timings.csv', index=False)
latency = inference.groupby(['model', 'seed'])['inference_ms_per_sample'].mean()
training_time = neural_history.groupby(['model', 'seed']).agg(
    epochs_run=('epoch', 'count'),
    train_s_per_epoch=('train_seconds', 'mean'),
    validation_s_per_epoch=('validation_seconds', 'mean'),
)
timings = runs[['model', 'seed', 'best_epoch', 'fit_seconds']].copy()
timings['parameters'] = timings['model'].map(parameter_counts)
timings = timings.join(training_time, on=['model', 'seed']).join(latency, on=['model', 'seed'])
timings.to_csv(output_dir / 'runtime_by_seed.csv', index=False)
report = timings.drop(columns='seed').groupby('model', sort=False).mean(numeric_only=True)
report.to_csv(output_dir / 'runtime_summary.csv')
neural_history[['model', 'seed', 'epoch', 'train_seconds', 'validation_seconds', 'epoch_seconds']].to_csv(
    output_dir / 'epoch_timings.csv', index=False,
)
display(timings)
display(report)


## 6. Tải checkpoint, kết quả và biểu đồ

In [ ]:
archive = shutil.make_archive(str(output_dir), 'zip', output_dir)
display(FileLink(archive))
